# 03 · A language model, one token at a time

A language model does one thing: given a text, it computes a score for every token of its vocabulary, saying how well that token would continue the text. Everything else (chatting, writing, translating) is this step in a loop.

In this notebook you will

1. see how text becomes tokens,
2. look at the model's scores for the next token,
3. **write the sampling step yourself** (temperature and top-k) in NumPy,
4. generate text with your own sampler.

The model is **DistilGPT-2**, a small GPT-2 (6 layers, 82 million parameters). **Download:** roughly 80 MB with 8-bit weights and roughly 330 MB with full precision; which one is used depends on the device. The browser caches it.

In [ ]:
import numpy as np
import lab

MODEL = "Xenova/distilgpt2"          # try "Xenova/gpt2" for the full-size GPT-2
CHAT_MODEL = "meta-llama/Llama-3.1-8B-Instruct"   # only for the optional last cell (hosted by Hugging Face)
PROMPT = "The capital of France is"

## 1. Tokens

The model never sees letters. A **tokenizer** cuts the text into pieces from a fixed vocabulary (50 257 pieces for GPT-2) and replaces each piece by its number. Frequent words are one token, rare words are several.

In GPT-2's vocabulary, the character `Ġ` at the beginning of a token stands for the space in front of the word. So `" Paris"` and `"Paris"` are different tokens.

In [ ]:
tok = await lab.tokenizer(MODEL)

text = "Tokenization splits unbelievably long words."
ids = tok.encode(text)
lab.table([{"position": i, "id": token_id, "token": token} for i, (token_id, token) in enumerate(zip(ids, tok.tokens(text)))])
print(len(text), "characters became", len(ids), "tokens")
print("decoded again:", repr(tok.decode(ids)))

## 2. The next token

`lab.next_token_probs` runs the model on the prompt and returns the most likely next tokens with their probabilities. (The first call downloads the model.)

In [ ]:
top = await lab.next_token_probs(MODEL, PROMPT, top_k=10)
lab.table([{"token": repr(token), "probability": round(p, 4)} for token, p in top])

Where do these probabilities come from? The model's last layer produces one real number per vocabulary entry, the **logits** $z \in \mathbb{R}^V$. `lab.next_token_logits` returns them as a NumPy array.

In [ ]:
logits = await lab.next_token_logits(MODEL, PROMPT)

print("shape:", logits.shape, " dtype:", logits.dtype)
print("largest logit:", float(logits.max()), "for token", repr(tok.decode([int(logits.argmax())])))
print("smallest logit:", float(logits.min()))

## 3. Your turn: softmax with temperature

The **softmax** turns logits into probabilities. With a temperature $T > 0$:

$$p_i = \frac{\exp(z_i / T)}{\sum_{j=1}^{V} \exp(z_j / T)}$$

- $T = 1$ is the distribution the model was trained to produce.
- $T < 1$ sharpens it: likely tokens become even more likely. As $T \to 0$, all probability goes to the largest logit.
- $T > 1$ flattens it: unlikely tokens get more chances.

One practical point: $\exp(1000)$ overflows. Subtracting the largest value from all $z_i / T$ before the exponential changes nothing in the result (numerator and denominator are multiplied by the same factor) and keeps every exponent at or below 0.

In [ ]:
def softmax(logits, temperature=1.0):
    """Probabilities from logits (a 1-D array), with temperature. Must not overflow for large logits."""
    z = np.asarray(logits, dtype=np.float64) / temperature
    # TODO: subtract the maximum, exponentiate, divide by the sum.
    raise NotImplementedError("Complete softmax, then run this cell and the check below.")

In [ ]:
p = softmax(np.array([1.0, 2.0, 3.0]))
assert np.allclose(p, [0.09003057, 0.24472847, 0.66524096]), p
assert np.isclose(softmax(logits).sum(), 1.0)
assert np.allclose(softmax(np.array([1000.0, 1001.0])), [0.26894142, 0.73105858]), "large logits must not overflow"
assert softmax(np.array([1.0, 2.0]), temperature=0.5)[1] > softmax(np.array([1.0, 2.0]))[1], "a low temperature sharpens"
assert softmax(np.array([1.0, 2.0]), temperature=2.0)[1] < softmax(np.array([1.0, 2.0]))[1], "a high temperature flattens"
assert np.allclose(softmax(np.array([1.0, 2.0]), temperature=0.5), softmax(np.array([2.0, 4.0]))), "dividing by T = 0.5 doubles the logits"
# Your probabilities for the real model agree with the table above:
best_token, best_p = top[0]
assert tok.decode([int(np.argmax(softmax(logits)))]) == best_token
assert np.isclose(softmax(logits).max(), best_p, rtol=1e-4)
print("softmax is correct.")

## 4. Your turn: top-k sampling

Always taking the most likely token (**greedy** decoding) gives dull, repetitive text. Sampling from the whole distribution sometimes picks a very unlikely token and derails. **Top-k sampling** is a compromise:

1. keep only the $k$ largest logits (set all others to $-\infty$, so their probability becomes 0),
2. apply the softmax with temperature to what is left,
3. draw one token at random with these probabilities.

Complete `sample_next`. Useful NumPy: `np.argsort`, `np.full_like`, `-np.inf`, and `rng.choice(len(p), p=p)`.

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, rng=None):
    """Draw the id of the next token. top_k=None means: sample from the whole vocabulary."""
    rng = rng or np.random.default_rng()
    logits = np.asarray(logits, dtype=np.float64)
    # TODO: 1. if top_k is given, set every logit outside the top_k largest to -np.inf
    #       2. p = softmax(..., temperature)
    #       3. return int(rng.choice(len(p), p=p))
    raise NotImplementedError("Complete sample_next, then run this cell and the check below.")

In [ ]:
rng = np.random.default_rng(0)
toy = np.array([2.0, 1.0, 0.5, -1.0, 3.0])

assert all(sample_next(toy, top_k=1, rng=rng) == 4 for _ in range(20)), "top_k=1 is greedy: always the largest logit"
assert {sample_next(toy, top_k=2, rng=rng) for _ in range(200)} == {0, 4}, "top_k=2 only ever returns the two best tokens"
assert all(sample_next(toy, temperature=0.01, rng=rng) == 4 for _ in range(20)), "a very low temperature is almost greedy"

draws = np.array([sample_next(toy, rng=rng) for _ in range(4000)])
frequencies = np.bincount(draws, minlength=len(toy)) / len(draws)
assert np.allclose(frequencies, softmax(toy), atol=0.03), "without top_k, tokens appear as often as softmax says"

draws = np.array([sample_next(toy, temperature=2.0, top_k=3, rng=rng) for _ in range(4000)])
frequencies = np.bincount(draws, minlength=len(toy)) / len(draws)
assert np.allclose(frequencies[[0, 1, 4]], softmax(toy[[0, 1, 4]], temperature=2.0), atol=0.03), "top_k and temperature together"
assert isinstance(sample_next(logits, top_k=40, rng=rng), int)
print("sample_next is correct.")

## 5. Generate text with your sampler

Generation is a loop: compute the logits for the text so far, sample a token, append it, repeat.

This loop runs the whole model again for every new token. Real implementations keep the intermediate results of the earlier tokens (the "key-value cache") so that each step only processes the newest token. Here, clarity wins over speed.

In [ ]:
async def generate(prompt, steps=15, temperature=1.0, top_k=None, seed=0):
    rng = np.random.default_rng(seed)
    ids = tok.encode(prompt)
    for _ in range(steps):
        step_logits = await lab.next_token_logits(MODEL, tok.decode(ids))
        ids.append(sample_next(step_logits, temperature=temperature, top_k=top_k, rng=rng))
    return tok.decode(ids)

print("greedy:          ", await generate(PROMPT, top_k=1))
print("T=0.7, top_k=40: ", await generate(PROMPT, temperature=0.7, top_k=40))
print("T=1.5, no top_k: ", await generate(PROMPT, temperature=1.5))

## 6. The same with the built-in generator

Transformers.js has this loop built in (with the cache, so it is faster). The options have the same meaning as in your function.

In [ ]:
generator = await lab.pipeline("text-generation", MODEL)
print("running on", generator.device)

out = await generator(PROMPT, max_new_tokens=30, do_sample=True, temperature=0.7, top_k=40)
print(out[0]["generated_text"])

## Optional: a large model, hosted

DistilGPT-2 is from 2019 and tiny. Models that are thousands of times larger do not fit into a browser tab, but the loop is the same. If you have a Hugging Face account, press **HF token** at the top of the page, paste a token, and run the next cell: it sends the question to a model hosted by Hugging Face's inference providers. The token stays in your browser and is not saved in the notebook.

In [ ]:
if lab.hf_token():
    answer = await lab.inference.chat(CHAT_MODEL, "In two sentences: what does the temperature do when a language model samples the next token?", max_tokens=120, temperature=0.7)
    print(answer)
else:
    print("No Hugging Face token is set, so this optional cell was skipped.")

## Questions

1. In the token table of section 1: which words were split, and into what? Try your own name.
2. Generate with `temperature=0.2` and with `temperature=2.0`. Describe the difference in one sentence each.
3. **Top-p** (nucleus) sampling keeps the smallest set of tokens whose probabilities add up to at least $p$. Write `sample_top_p(logits, p=0.9, rng=None)`. When does it keep more tokens than top-k with $k = 40$, and when fewer?